# TruthfulQA (LM-Polygraph) Continuation: Exploration

This notebook explores the Hugging Face parquet at:

- `hf://datasets/LM-Polygraph/truthfulqa/continuation/test-00000-of-00001.parquet`

Notes:
- If access is gated, log in first in a terminal with `huggingface-cli login`, or set `HF_TOKEN` in your environment.
- The dataset has two columns: `input` (few-shot prompt) and `output` (NumPy array of candidate continuations).


In [ ]:
# If you haven't installed deps yet, run this in a terminal (recommended):
#   python3 -m venv .venv
#   .venv/bin/pip install -U pip
#   .venv/bin/pip install pandas pyarrow fsspec huggingface_hub
#
# Optional (to run Jupyter from the same env):
#   .venv/bin/pip install jupyter ipykernel
#   .venv/bin/python -m ipykernel install --user --name nlphackathon --display-name "NLPHackathon (.venv)"

import pandas as pd
import numpy as np
from collections import Counter

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

DATA_PATH = "hf://datasets/LM-Polygraph/truthfulqa/continuation/test-00000-of-00001.parquet"
DATA_PATH

In [ ]:
df = pd.read_parquet(DATA_PATH)
df.shape

In [ ]:
df.head(3)

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

## Understand `output`

`output` is an `np.ndarray` per row (candidate continuations). That means operations like `nunique()` need a small adjustment if you want to treat each array as hashable.


In [ ]:
type(df.loc[0, "output"]), df.loc[0, "output"][:5]

In [ ]:
Counter(type(x).__name__ for x in df["output"]).most_common()

## Extract the final question from `input`

Each `input` is a few-shot prompt ending with a final `Q: ...` and an `A:` that is empty (the model is supposed to continue).


In [ ]:
def extract_question(prompt: str) -> str:
    marker = "\n\nQ: "
    idx = prompt.rfind(marker)
    start = idx + len(marker) if idx != -1 else prompt.rfind("Q: ") + 3
    end = prompt.rfind("\nA:")
    return prompt[start:end].strip() if end != -1 else prompt[start:].strip()


questions = df["input"].map(extract_question)
questions.head(10)

In [ ]:
questions.str.len().describe()

## Candidate answer counts


In [ ]:
choice_counts = df["output"].map(len)
choice_counts.describe()

In [ ]:
choice_counts.value_counts().sort_index()

## Flattened answer text stats


In [ ]:
all_choices = [str(choice) for arr in df["output"] for choice in arr]
choice_len = pd.Series([len(s) for s in all_choices])
choice_len.describe()

## Build a parsed table

This is often more convenient for analysis: one row per original example, with an explicit `question` and Python `list` of choices.


In [ ]:
parsed = pd.DataFrame(
    {
        "question": questions,
        "choices": df["output"].map(lambda arr: [str(x) for x in arr]),
        "num_choices": choice_counts,
        "prompt": df["input"],
    }
)
parsed.head(3)

## Long-form (exploded) table

One row per candidate answer choice, useful for filtering and counting.


In [ ]:
long = parsed[["question", "choices"]].explode("choices", ignore_index=True)
long = long.rename(columns={"choices": "choice"})
long.head(10)

In [ ]:
long["choice_len"] = long["choice"].astype(str).str.len()
long["choice_len"].describe()

## Quick interactive poking

Pick a row index and inspect its question + candidate answers.


In [ ]:
idx = 0
print(parsed.loc[idx, "question"])
print("\nChoices:")
for i, c in enumerate(parsed.loc[idx, "choices"], start=1):
    print(f"{i}. {c}")

## (Optional) Save parsed outputs

Write the parsed tables to disk so you can reuse them quickly without re-reading from `hf://...`.


In [ ]:
# parsed.to_parquet("truthfulqa_continuation_parsed.parquet", index=False)
# long.to_parquet("truthfulqa_continuation_long.parquet", index=False)

# Or CSV (bigger):
# parsed.to_csv("truthfulqa_continuation_parsed.csv", index=False)
# long.to_csv("truthfulqa_continuation_long.csv", index=False)
None